In [45]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

from seaborn import set_style
set_style("whitegrid")

In [ ]:
# Source - https://stackoverflow.com/a/50051542
# Posted by Aaron Brock, modified by community. See post 'Timeline' for change history
# Retrieved 2026-02-26, License - CC BY-SA 3.0

def to_csv(df, path):
    # Prepend dtypes to the top of df (from https://stackoverflow.com/a/43408736/7607701)
    df.loc[-1] = df.dtypes
    df.index = df.index + 1
    df.sort_index(inplace=True)
    # Then save it to a csv
    df.to_csv(path, index=False)

def read_csv(path):
    # Read types first line of csv
    dtypes = {key:value for (key,value) in pd.read_csv(path,    
              nrows=1).iloc[0].to_dict().items() if 'date' not in value}

    parse_dates = [key for (key,value) in pd.read_csv(path, 
                   nrows=1).iloc[0].to_dict().items() if 'date' in value]
    # Read the rest of the lines with the types from above
    return pd.read_csv(path, dtype=dtypes, parse_dates=parse_dates, skiprows=[1])

In [ ]:
#Read in the combined weather/schedule/summary files for each sunday time period and combine them into a single dataframe 

loc = '../data/clean_data/'

csv_file_names = [loc+file for file in os.listdir(loc) if file.startswith('schedule_weather_sunday_period')]
list_of_dataframes = [read_csv(file) 
    for file in csv_file_names]
sun_df = pd.concat(list_of_dataframes, ignore_index=True)

In [ ]:
# List the features we want to keep, sort them by their dtypes

features = [
    'time_period',
    'bunch',
    'EB/WB', 
    'gap', 
    'No. of Veh', 
    'Avg. spd (km/h)',
    'Interruption',
    'conditions',
    'pop',     
    'pop_category',   
    'temperature',         
    'wind_direction',        
    'wind_speed'
    ]

# For the next (real) model, need to sort the types of features into numeric/categorical

In [49]:
X = sun_df[features]
y = sun_df['total delay']
groups = sun_df['date']

from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error

G_kfold = GroupKFold(n_splits=5)

maes = []
for fold, (train_index, test_index) in enumerate(G_kfold.split(X, y, groups=groups), 1):
    X_train, X_test = X.iloc[train_index].copy(), X.iloc[test_index].copy()
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]


    temp_train = pd.concat([X_train[["time_period"]], y_train.rename("total delay")], axis=1)
    mean_by_period = temp_train.groupby("time_period")['total delay'].mean()
    y_pred = X_test["time_period"].map(mean_by_period)

    mae = mean_absolute_error(y_test, y_pred)
    maes.append(mae)

    test_dates = sun_df["date"].iloc[test_index].nunique()
    print(f"Fold {fold}: test_dates={test_dates}, MAE={mae:.2f}")

print("Mean MAE:", np.mean(maes))


Fold 1: test_dates=10, MAE=1098.62
Fold 2: test_dates=10, MAE=979.81
Fold 3: test_dates=10, MAE=978.70
Fold 4: test_dates=10, MAE=1147.67
Fold 5: test_dates=10, MAE=1170.02
Mean MAE: 1074.96435
